# Model Evaluation Notebook

Quick-start evaluation of an exported threat screener bundle against new sequences. No training required.

**What you need:**
1. A bundle `.tar.gz` produced by the training notebook's export cell (PATCH H v8)
2. Either a FASTA file of sequences to score, or internet access to fetch from NCBI

**What this notebook does:**
1. Loads the model bundle
2. Imports sequences from disk (FASTA file) and/or from NCBI online
3. Scores all sequences through the production model
4. Imports two reference sets from NCBI (similar-distribution + external/out-of-distribution) for comparison
5. Produces a three-way generalization analysis with statistics and plots

**Cell order to run:** top to bottom. Configure paths at the top of cells 2, 3, 4 before running.

**Expected runtime:** under 5 minutes if you only score a local FASTA, 15-25 minutes if you fetch reference sets from NCBI.


## 1. Setup


In [ ]:
# Setup and imports

import os, sys, json, gzip, math, time, random, tarfile, tempfile
from pathlib import Path
from collections import defaultdict, Counter

try:
    import numpy as np
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy"])
    import numpy as np

try:
    import xgboost as xgb
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])
    import xgboost as xgb

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

try:
    from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
    from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score

print(f"xgboost {xgb.__version__}, numpy {np.__version__}")
print(f"working dir: {os.getcwd()}")


## 2. Load model bundle

Edit `BUNDLE_PATH` to point at the .tar.gz file produced by the training notebook's export cell. Default path is `Kmer_Flagger/outputs4/threat_screener_bundle.tar.gz`.


In [ ]:
# Load the model bundle

BUNDLE_PATH = "Insert_Path"
OUTPUT_DIR = "evaluation_outputs"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

if not Path(BUNDLE_PATH).exists():
    raise FileNotFoundError(
        f"BUNDLE_PATH does not exist: {BUNDLE_PATH}\n"
        f"Edit BUNDLE_PATH at the top of this cell to point at your .tar.gz bundle."
    )

_extract_root = Path(tempfile.mkdtemp(prefix='eval_bundle_'))
with tarfile.open(BUNDLE_PATH, 'r:gz') as tar:
    tar.extractall(_extract_root)
_bundle_dir = _extract_root / os.listdir(_extract_root)[0]
print(f"[bundle] extracted to {_bundle_dir}")
print(f"[bundle] contents:")
for f in sorted(os.listdir(_bundle_dir)):
    size = (_bundle_dir / f).stat().st_size
    print(f"   {f:<35} {size:>10} bytes")

_CODON = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','TCT':'S','TCC':'S','TCA':'S','TCG':'S',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','TGT':'C','TGC':'C','TGA':'*','TGG':'W',
    'CTT':'L','CTC':'L','CTA':'L','CTG':'L','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','ACT':'T','ACC':'T','ACA':'T','ACG':'T',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K','AGT':'S','AGC':'S','AGA':'R','AGG':'R',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'GAT':'D','GAC':'D','GAA':'E','GAG':'E','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}
_COMP = str.maketrans('ACGT', 'TGCA')

def _revcomp(s): return s.translate(_COMP)[::-1]

def _translate(s):
    return ''.join(_CODON.get(s[i:i+3], 'X') for i in range(0, len(s) - 2, 3))

def _triplet_entropy(window):
    if len(window) < 3: return 0.0
    c = Counter(window[i:i+3] for i in range(len(window)-2))
    total = sum(c.values())
    if total == 0: return 0.0
    h = 0.0
    for v in c.values():
        p = v / total
        if p > 0: h -= p * math.log2(p)
    return h

def dust_mask(seq, window=32, threshold=2.5):
    if len(seq) < window:
        return 'N' * len(seq) if _triplet_entropy(seq) < threshold else seq
    masked = list(seq)
    any_masked = False
    for i in range(len(seq) - window + 1):
        if _triplet_entropy(seq[i:i+window]) < threshold:
            for j in range(i, i + window):
                masked[j] = 'N'
            any_masked = True
    return ''.join(masked) if any_masked else seq


class ThreatScreenBundle:
    def __init__(self, model, dna_scores, prot_scores, dna_km2i, pkm2i,
                 t_long, i_long, thresholds, metadata):
        self.model = model
        self.dna_scores = dna_scores
        self.prot_scores = prot_scores
        self.dna_km2i = dna_km2i
        self.pkm2i = pkm2i
        self.t_long = t_long
        self.i_long = i_long
        self.thresholds = thresholds
        self.metadata = metadata
        self._n_dna = len(dna_km2i)
        self._n_prot = len(pkm2i)
        self._k_dna = metadata['k_dna']
        self._k_prot = metadata['k_prot']
        self._k_long = metadata['k_long']
        self._applies_dust = metadata['applies_dust_masking']
        self._rev_screen = metadata['rev_screen_enabled']

    @classmethod
    def load(cls, bundle_dir):
        d = Path(bundle_dir)
        with open(d / 'metadata.json') as f:
            metadata = json.load(f)
        model = xgb.Booster()
        model.load_model(str(d / 'model.json'))
        with open(d / 'dna_scores.json') as f:
            dna_scores = json.load(f)
        with open(d / 'prot_scores.json') as f:
            prot_scores = json.load(f)
        with open(d / 'dna_km2i.json') as f:
            dna_km2i = json.load(f)
        with open(d / 'pkm2i.json') as f:
            pkm2i = json.load(f)
        t_long = i_long = None
        tl_path = d / 'threat_longkmers.txt.gz'
        il_path = d / 'innocuous_longkmers.txt.gz'
        if tl_path.exists():
            with gzip.open(tl_path, 'rt') as f:
                t_long = set(line.strip() for line in f if line.strip())
        if il_path.exists():
            with gzip.open(il_path, 'rt') as f:
                i_long = set(line.strip() for line in f if line.strip())
        with open(d / 'thresholds.json') as f:
            thresholds = json.load(f)
        return cls(model, dna_scores, prot_scores, dna_km2i, pkm2i,
                   t_long, i_long, thresholds, metadata)

    def _feat_vec(self, seq):
        if self._applies_dust:
            seq = dust_mask(seq)
        vd = np.zeros(self._n_dna, dtype=np.float32)
        vp = np.zeros(self._n_prot, dtype=np.float32)
        k = self._k_dna
        if len(seq) >= k:
            for i in range(len(seq) - k + 1):
                km = seq[i:i+k]
                if 'N' in km: continue
                idx = self.dna_km2i.get(km)
                if idx is not None: vd[idx] += 1
            vd = vd / max(1, len(seq) - k + 1)
        kp = self._k_prot
        for fs in (seq[0:], seq[1:], seq[2:]):
            p = _translate(fs)
            for i in range(len(p) - kp + 1):
                pk = p[i:i+kp]
                if 'X' in pk or '*' in pk: continue
                idx = self.pkm2i.get(pk)
                if idx is not None: vp[idx] += 1
        rc = _revcomp(seq)
        for fs in (rc[0:], rc[1:], rc[2:]):
            p = _translate(fs)
            for i in range(len(p) - kp + 1):
                pk = p[i:i+kp]
                if 'X' in pk or '*' in pk: continue
                idx = self.pkm2i.get(pk)
                if idx is not None: vp[idx] += 1
        return np.concatenate([vd, vp])

    def _reverse_screen(self, seq, base_prob):
        if not self._rev_screen or self.t_long is None:
            return base_prob
        if len(seq) < self._k_long:
            return base_prob
        th = ih = 0
        for i in range(len(seq) - self._k_long + 1):
            km = seq[i:i+self._k_long]
            if km in self.t_long: th += 1
            if km in self.i_long: ih += 1
        if th + ih == 0:
            return base_prob
        return math.sqrt(float(base_prob) * max(0.001, th / (th + ih)))

    def score(self, seq):
        seq = seq.upper()
        X = np.array([self._feat_vec(seq)], dtype=np.float32)
        p_raw = float(self.model.predict(xgb.DMatrix(X))[0])
        return self._reverse_screen(seq, p_raw)

    def score_batch(self, seqs):
        seqs = [s.upper() for s in seqs]
        X = np.array([self._feat_vec(s) for s in seqs], dtype=np.float32)
        raw = self.model.predict(xgb.DMatrix(X))
        return np.array([self._reverse_screen(seqs[i], raw[i]) for i in range(len(seqs))])


bundle = ThreatScreenBundle.load(_bundle_dir)
print(f"\n[bundle] loaded successfully")
print(f"[bundle] winner_condition:    {bundle.metadata['winner_condition']}")
print(f"[bundle] xgboost (training):  {bundle.metadata['xgboost_version']}")
print(f"[bundle] xgboost (this env):  {xgb.__version__}")
print(f"[bundle] features:            {bundle.metadata['n_dna_kmers']} DNA + "
      f"{bundle.metadata['n_prot_kmers']} protein = {bundle.metadata['n_total_features']}")
print(f"[bundle] applies_dust:        {bundle.metadata['applies_dust_masking']}")
print(f"[bundle] rev_screen_enabled:  {bundle.metadata['rev_screen_enabled']}")

print(f"\n[bundle] sanity scoring on a few test sequences:")
test_seqs = [
    "ATGAAAGCCATTGCCGCCTGGGGCTGCAAGAACGATGACGAT",
    "GCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGC",
    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA",
]
for s in test_seqs:
    print(f"   score={bundle.score(s):.4f}  seq={s[:40]}{'...' if len(s) > 40 else ''}")


## 3. Load sequences from FASTA file (optional)

Set `USER_FASTA_PATH` to a `.fasta` or `.fasta.gz` file. Headers may use the training-notebook format (`>id|category=threat|source=...`) or any standard FASTA format - sequences without explicit category metadata are treated as unlabeled and excluded from AUC computation, but still scored.

If you have no FASTA file and only want online evaluation, leave `USER_FASTA_PATH = ''` and run cell 4 with `ONLINE_FETCH_USER_SEQS = True`.


In [ ]:
# Load sequences from a FASTA file (offline)

USER_FASTA_PATH = ""

def load_sequences_from_fasta(path):
    """Read a FASTA file (gzipped or plain). Returns list of dicts.
    Header conventions supported:
        >id|category=threat|source=foo|...      (training-notebook bundle format)
        >id [organism]                           (NCBI default)
        >id  (anything)                          (treated as category=unknown)"""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"FASTA file not found: {path}")
    opener = gzip.open if str(path).endswith('.gz') else open
    out = []
    current = None
    with opener(path, 'rt') as f:
        for line in f:
            line = line.rstrip()
            if line.startswith('>'):
                if current is not None:
                    current['sequence'] = ''.join(current['_chunks'])
                    del current['_chunks']
                    out.append(current)
                hdr = line[1:].strip()
                parts = hdr.split('|')
                seq_id = parts[0].split()[0]
                meta = {}
                for kv in parts[1:]:
                    if '=' in kv:
                        k, _, v = kv.partition('=')
                        meta[k.strip()] = v.strip()
                current = {
                    'seq_id': seq_id,
                    'category': meta.get('category', 'unknown'),
                    'source': meta.get('source', 'user_fasta'),
                    'description': hdr,
                    '_chunks': [],
                }
            elif current is not None:
                current['_chunks'].append(line.strip().upper())
    if current is not None:
        current['sequence'] = ''.join(current['_chunks'])
        del current['_chunks']
        out.append(current)
    return out


user_seqs = []
if USER_FASTA_PATH and Path(USER_FASTA_PATH).exists():
    user_seqs = load_sequences_from_fasta(USER_FASTA_PATH)
    print(f"[fasta] loaded {len(user_seqs)} sequences from {USER_FASTA_PATH}")

    cats = Counter(s['category'] for s in user_seqs)
    srcs = Counter(s['source'] for s in user_seqs)
    print(f"[fasta] categories: {dict(cats)}")
    print(f"[fasta] sources:    {dict(srcs)}")

    if user_seqs:
        sample = ''.join(s['sequence'] for s in user_seqs[:50]).upper()
        if sample:
            dna_frac = sum(1 for c in sample if c in 'ACGTN') / len(sample)
            if dna_frac < 0.95:
                print(f"\n  WARNING: sequences look non-DNA ({dna_frac:.1%} A/C/G/T/N). "
                       f"The model expects DNA. Scoring will be unreliable.")
            else:
                print(f"  DNA sanity check: {dna_frac:.1%} A/C/G/T/N (OK)")
else:
    if USER_FASTA_PATH:
        print(f"[fasta] WARNING: USER_FASTA_PATH set but file does not exist: {USER_FASTA_PATH}")
    else:
        print(f"[fasta] no FASTA path set (USER_FASTA_PATH is empty); skipping offline import")
        print(f"[fasta] to use offline mode, set USER_FASTA_PATH at the top of this cell")


## 4. Fetch reference sequences from NCBI (online)

Always fetches a **similar-distribution** reference set and an **external (out-of-distribution)** reference set for comparison.

Optionally also fetches a user evaluation set if `ONLINE_FETCH_USER_SEQS=True` (useful when no FASTA file is available).

Sequences are deduplicated against `NCBI_CACHE_DIR/seen_accessions.json` so repeated runs return fresh accessions, not the same top hits.

**Default targets are small for fast first-run testing.** Bump them up (e.g. 500 / 1000) for production-quality statistics.

Wall time: ~1-2 minutes per 100 sequences (NCBI rate-limited at 0.4s/query).


In [ ]:
# Online NCBI sequence importer

ONLINE_FETCH_USER_SEQS = False
USER_THREAT_TARGET = 50
USER_BENIGN_TARGET = 100

SIMILAR_THREAT_TARGET = 100
SIMILAR_BENIGN_TARGET = 200
EXTERNAL_THREAT_TARGET = 100
EXTERNAL_BENIGN_TARGET = 200

NCBI_CACHE_DIR = "insert_path/ncbi_cache"
NCBI_EMAIL = ""

try:
    from Bio import Entrez, SeqIO
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "biopython"])
    from Bio import Entrez, SeqIO

Entrez.email = NCBI_EMAIL


SIMILAR_THREAT_QUERIES = [
    "(virulence factor) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(toxin) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(hemolysin) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(enterotoxin) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(cytotoxin) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(secretion system) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Salmonella[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Escherichia coli[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Staphylococcus aureus[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Streptococcus[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Clostridium[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Listeria[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Yersinia[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Vibrio[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Pseudomonas aeruginosa[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Bacillus anthracis[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
]
SIMILAR_BENIGN_QUERIES = [
    "(housekeeping gene) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(ribosomal protein) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(DNA polymerase) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(RNA polymerase) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Bacillus subtilis[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Lactobacillus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Streptomyces[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Caulobacter crescentus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
]

EXTERNAL_THREAT_QUERIES = [
    "Borrelia burgdorferi[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Treponema pallidum[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Leptospira interrogans[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Chlamydia trachomatis[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Mycoplasma pneumoniae[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Aeromonas[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Photorhabdus[Organism] AND toxin AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Bartonella[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
]
EXTERNAL_BENIGN_QUERIES = [
    "Acidobacterium[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Verrucomicrobia[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Aquifex[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Halobacterium[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Sulfolobus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Pyrococcus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "environmental sample[Organism] AND bacteria AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "uncultured bacterium[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
]

DEDUP_LOG_FILENAME = 'seen_accessions.json'


def _load_seen(cache_dir):
    p = Path(cache_dir) / DEDUP_LOG_FILENAME
    if not p.exists():
        return defaultdict(set)
    with open(p) as f:
        raw = json.load(f)
    return defaultdict(set, {k: set(v) for k, v in raw.items()})


def _save_seen(cache_dir, seen):
    p = Path(cache_dir) / DEDUP_LOG_FILENAME
    p.write_text(json.dumps({k: sorted(v) for k, v in seen.items()}, indent=2))


def fetch_ncbi(category, target_count, queries, cache_dir, log_fn=print):
    """Fetch NCBI nucleotide sequences with persistent dedup. Returns list of dicts."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    seen = _load_seen(cache_dir)
    log_fn(f"[ncbi {category}] target={target_count}, queries={len(queries)}, "
           f"prior accessions in cache: {sum(len(v) for v in seen.values())}")
    per_query = max(target_count // len(queries), 5)
    out = []
    for q_idx, query in enumerate(queries):
        if len(out) >= target_count: break
        seen_for_q = seen[query]
        extra = max(2, 1 + len(seen_for_q) // max(1, per_query))
        try:
            handle = Entrez.esearch(db="nucleotide", term=query,
                                     retmax=per_query * extra,
                                     retstart=0, sort="relevance")
            result = Entrez.read(handle); handle.close()
        except Exception as e:
            log_fn(f"  esearch failed [{q_idx+1}/{len(queries)}]: {e}")
            continue
        ids = result.get('IdList', [])
        fresh = [i for i in ids if i not in seen_for_q][:per_query]
        if not fresh:
            log_fn(f"  [{q_idx+1}/{len(queries)}] '{query[:50]}...' -> no new accessions")
            continue
        try:
            handle = Entrez.efetch(db="nucleotide", id=",".join(fresh),
                                    rettype="fasta", retmode="text")
            records = list(SeqIO.parse(handle, "fasta"))
            handle.close()
        except Exception as e:
            log_fn(f"  efetch failed [{q_idx+1}/{len(queries)}]: {e}")
            continue
        added = 0
        for rec in records:
            seq_str = str(rec.seq).upper()
            if len(seq_str) < 30 or len(seq_str) > 5000: continue
            valid = sum(1 for c in seq_str if c in 'ACGTN')
            if valid / len(seq_str) < 0.95: continue
            out.append({
                'seq_id': f"NCBI_{category}_{rec.id}",
                'category': category, 'source': f"NCBI_Eval_{category.capitalize()}",
                'description': str(rec.description)[:200],
                'sequence': seq_str,
            })
            seen[query].add(rec.id)
            added += 1
            if len(out) >= target_count: break
        log_fn(f"  [{q_idx+1}/{len(queries)}] '{query[:50]}...' +{added} (total: {len(out)})")
        _save_seen(cache_dir, seen)
        time.sleep(0.4)
    log_fn(f"[ncbi {category}] DONE: {len(out)} new sequences")
    return out


# Optionally fetch user sequences online (in addition to or instead of FASTA)
if ONLINE_FETCH_USER_SEQS:
    print("[user-online] fetching user evaluation set from NCBI")
    user_online_threats = fetch_ncbi('threat', USER_THREAT_TARGET,
                                      SIMILAR_THREAT_QUERIES,
                                      Path(NCBI_CACHE_DIR) / 'user_set')
    user_online_benigns = fetch_ncbi('benign', USER_BENIGN_TARGET,
                                      SIMILAR_BENIGN_QUERIES,
                                      Path(NCBI_CACHE_DIR) / 'user_set')
    user_seqs = (user_seqs if 'user_seqs' in dir() else []) + user_online_threats + user_online_benigns
    print(f"[user-online] user_seqs now contains {len(user_seqs)} sequences total")
else:
    print(f"[user-online] online user-set fetch disabled (set ONLINE_FETCH_USER_SEQS=True to enable)")

print(f"\n[ncbi] fetching SIMILAR-distribution reference set "
      f"(threats={SIMILAR_THREAT_TARGET}, benign={SIMILAR_BENIGN_TARGET})")
similar_threats = fetch_ncbi('threat', SIMILAR_THREAT_TARGET,
                              SIMILAR_THREAT_QUERIES,
                              Path(NCBI_CACHE_DIR) / 'similar')
similar_benigns = fetch_ncbi('benign', SIMILAR_BENIGN_TARGET,
                              SIMILAR_BENIGN_QUERIES,
                              Path(NCBI_CACHE_DIR) / 'similar')
similar_seqs = similar_threats + similar_benigns

print(f"\n[ncbi] fetching EXTERNAL (out-of-distribution) reference set "
      f"(threats={EXTERNAL_THREAT_TARGET}, benign={EXTERNAL_BENIGN_TARGET})")
external_threats = fetch_ncbi('threat', EXTERNAL_THREAT_TARGET,
                               EXTERNAL_THREAT_QUERIES,
                               Path(NCBI_CACHE_DIR) / 'external')
external_benigns = fetch_ncbi('benign', EXTERNAL_BENIGN_TARGET,
                               EXTERNAL_BENIGN_QUERIES,
                               Path(NCBI_CACHE_DIR) / 'external')
external_seqs = external_threats + external_benigns

print(f"\n[ncbi] all reference sets ready:")
print(f"   user-supplied: {len(user_seqs) if 'user_seqs' in dir() else 0}")
print(f"   similar:       {len(similar_seqs)}")
print(f"   external:      {len(external_seqs)}")


## 5. Analysis

Fragments all loaded parents at lengths [20, 30, 50, 100, 200] bp, scores every fragment through the production model, then reports:

- **Length-bucketed AUC** for each set
- **Score statistics** (mean, median, range) per set
- **AT-bias check** (does the model over-flag low-GC benign?)
- **4-panel plot**: AUC bars, benign score distributions, ROC curves, GC-vs-score scatter
- **CSV outputs** in `evaluation_outputs/`: per-bucket summary and per-fragment scores


In [ ]:
# Three-way analysis

EVAL_FRAGMENT_LENGTHS = [20, 30, 50, 100, 200]
FRAGS_PER_PARENT_PER_LEN = 3
LENGTH_BUCKETS = [(15, 30), (30, 60), (60, 100), (100, 200), (200, 10000)]

def _gc(seq):
    g = seq.count('G') + seq.count('C')
    n = seq.count('N')
    return g / max(1, len(seq) - n)

def fragment_and_score(parent_seq_dicts, set_name, rng_seed):
    """Fragment parent sequences into fixed lengths and score them."""
    rng = random.Random(rng_seed)
    seqs, labels, lens, gcs, parent_ids = [], [], [], [], []
    for sd in parent_seq_dicts:
        parent_seq = sd['sequence']
        if len(parent_seq) < 30: continue
        if sd.get('category') == 'threat':
            label = 1
        elif sd.get('category') == 'benign':
            label = 0
        else:
            label = -1
        for L in EVAL_FRAGMENT_LENGTHS:
            if len(parent_seq) < L: continue
            for _ in range(FRAGS_PER_PARENT_PER_LEN):
                start = rng.randrange(0, len(parent_seq) - L + 1)
                sub = parent_seq[start:start+L]
                seqs.append(sub)
                labels.append(label)
                lens.append(L)
                gcs.append(_gc(sub))
                parent_ids.append(sd['seq_id'])
    if not seqs:
        return {'name': set_name, 'n': 0}
    print(f"  [{set_name}] scoring {len(seqs)} fragments from "
          f"{len(parent_seq_dicts)} parents ...")
    t0 = time.time()
    scores = bundle.score_batch(seqs)
    print(f"  [{set_name}] scored in {time.time()-t0:.1f}s")
    return {
        'name': set_name, 'n': len(seqs),
        'seqs': seqs, 'labels': np.array(labels),
        'lens': np.array(lens), 'gcs': np.array(gcs),
        'parent_ids': parent_ids, 'scores': scores,
    }

print(f"[analysis 1/3] scoring all sequence sets through the model\n")
results = []
if 'user_seqs' in dir() and user_seqs:
    user_res = fragment_and_score(user_seqs, 'user', 1001)
    results.append(user_res)
if 'similar_seqs' in dir() and similar_seqs:
    sim_res = fragment_and_score(similar_seqs, 'similar', 2002)
    results.append(sim_res)
if 'external_seqs' in dir() and external_seqs:
    ext_res = fragment_and_score(external_seqs, 'external', 3003)
    results.append(ext_res)

if not results:
    raise RuntimeError("No sequences to analyze. Provide a FASTA in cell 3 "
                        "or enable online fetch in cell 4.")

print(f"\n[analysis 2/3] computing statistics\n")

def _bucket_metrics(res, lo, hi):
    if res['n'] == 0: return None
    mask = (res['lens'] >= lo) & (res['lens'] < hi)
    valid = mask & (res['labels'] >= 0)
    if valid.sum() < 30 or len(np.unique(res['labels'][valid])) < 2:
        return None
    y = res['labels'][valid]; sc = res['scores'][valid]
    auc = roc_auc_score(y, sc)
    fpr, tpr, _ = roc_curve(y, sc)
    return {
        'n': int(valid.sum()), 'n_threat': int(y.sum()),
        'n_benign': int((y == 0).sum()),
        'auc': float(auc),
        'ap': float(average_precision_score(y, sc)),
        's_at_5': float(np.interp(0.05, fpr, tpr)),
        's_at_1': float(np.interp(0.01, fpr, tpr)),
    }

print(f"  Length-bucketed AUC across all sets:\n")
header = f"  {'bucket':<14}" + "".join(f" {r['name']+' AUC':>14}" for r in results)
print(header)

bucket_rows = []
for lo, hi in LENGTH_BUCKETS:
    label_hi = 'inf' if hi >= 10000 else str(hi - 1)
    bkt = f"{lo}-{label_hi}bp"
    metrics_for_buckets = [_bucket_metrics(r, lo, hi) for r in results]
    bucket_rows.append((bkt, metrics_for_buckets))
    cells = []
    for m in metrics_for_buckets:
        cells.append(f"{m['auc']:.3f} (n={m['n']})" if m else "-")
    print(f"  {bkt:<14}" + "".join(f" {c:>14}" for c in cells))

print(f"\n  Score statistics (mean and median, all lengths pooled):\n")
print(f"  {'set':<14} {'n':>6} {'mean':>8} {'median':>8} {'min':>6} {'max':>6}  "
       f"{'mean_threat':>12} {'mean_benign':>12}")
for r in results:
    if r['n'] == 0: continue
    sc = r['scores']
    threat_mask = r['labels'] == 1
    benign_mask = r['labels'] == 0
    threat_mean = float(sc[threat_mask].mean()) if threat_mask.sum() else float('nan')
    benign_mean = float(sc[benign_mask].mean()) if benign_mask.sum() else float('nan')
    print(f"  {r['name']:<14} {r['n']:>6} {float(sc.mean()):>8.3f} "
          f"{float(np.median(sc)):>8.3f} {float(sc.min()):>6.3f} {float(sc.max()):>6.3f}  "
          f"{threat_mean:>12.3f} {benign_mean:>12.3f}")

print(f"\n  AT-bias check on benign sequences:\n")
print(f"  {'set':<14} {'AT mean':>10} {'AT n':>6} {'normal mean':>12} {'normal n':>10} {'delta':>8}")
for r in results:
    if r['n'] == 0: continue
    b_at = (r['labels'] == 0) & (r['gcs'] < 0.35)
    b_no = (r['labels'] == 0) & (r['gcs'] >= 0.35)
    if b_at.sum() < 10 or b_no.sum() < 10:
        print(f"  {r['name']:<14} insufficient (AT n={int(b_at.sum())}, normal n={int(b_no.sum())})")
        continue
    m_at = float(r['scores'][b_at].mean())
    m_no = float(r['scores'][b_no].mean())
    print(f"  {r['name']:<14} {m_at:>10.3f} {int(b_at.sum()):>6} "
           f"{m_no:>12.3f} {int(b_no.sum()):>10} {m_at-m_no:>+8.3f}")


print(f"\n[analysis 3/3] writing summary and plots\n")
PLOT_DIR = Path(OUTPUT_DIR) / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

summary_csv = Path(OUTPUT_DIR) / "evaluation_summary.csv"
with open(summary_csv, 'w') as f:
    f.write('bucket,set,n,n_threat,n_benign,auc,avg_precision,sens_at_fpr_5pct,sens_at_fpr_1pct\n')
    for bkt, metrics_list in bucket_rows:
        for r, m in zip(results, metrics_list):
            if m:
                f.write(f"{bkt},{r['name']},{m['n']},{m['n_threat']},{m['n_benign']},"
                        f"{m['auc']:.4f},{m['ap']:.4f},{m['s_at_5']:.4f},{m['s_at_1']:.4f}\n")
print(f"  wrote {summary_csv}")

scores_csv = Path(OUTPUT_DIR) / "all_scores.csv"
with open(scores_csv, 'w') as f:
    f.write('set,parent_id,length,gc,label,score\n')
    for r in results:
        if r['n'] == 0: continue
        for i in range(r['n']):
            lbl = 'threat' if r['labels'][i] == 1 else ('benign' if r['labels'][i] == 0 else 'unknown')
            f.write(f"{r['name']},{r['parent_ids'][i]},{int(r['lens'][i])},"
                     f"{r['gcs'][i]:.3f},{lbl},{float(r['scores'][i]):.4f}\n")
print(f"  wrote {scores_csv}")

colors = {'user': 'steelblue', 'similar': 'mediumseagreen', 'external': 'darkorange'}
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

ax = axes[0, 0]
buckets_to_plot = []
auc_lists = {r['name']: [] for r in results}
for bkt, metrics_list in bucket_rows:
    if any(m is not None for m in metrics_list):
        buckets_to_plot.append(bkt)
        for r, m in zip(results, metrics_list):
            auc_lists[r['name']].append(m['auc'] if m else 0)
if buckets_to_plot:
    x = np.arange(len(buckets_to_plot))
    n_sets = len(results)
    w = 0.8 / max(n_sets, 1)
    for i, r in enumerate(results):
        offset = (i - (n_sets - 1) / 2) * w
        vals = auc_lists[r['name']]
        ax.bar(x + offset, vals, w, color=colors.get(r['name'], 'gray'),
               label=r['name'], alpha=0.85)
        for j, v in enumerate(vals):
            if v > 0:
                ax.text(x[j] + offset, v + 0.01, f'{v:.2f}', ha='center', fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(buckets_to_plot)
    ax.set_ylabel('AUC')
    ax.set_ylim(0, 1.05)
    ax.set_title('AUC by length bucket')
    ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')
    ax.axhline(0.5, ls='--', color='gray', alpha=0.5, lw=1)
else:
    ax.text(0.5, 0.5, 'no labeled data with discriminable buckets\n'
            '(may be unlabeled FASTA + no NCBI fetch)',
            ha='center', va='center', transform=ax.transAxes)
    ax.set_title('AUC by length bucket')

ax = axes[0, 1]
bins = np.linspace(0, 1, 41)
for r in results:
    if r['n'] == 0: continue
    benign_scores = r['scores'][r['labels'] == 0]
    if len(benign_scores):
        ax.hist(benign_scores, bins=bins, alpha=0.4,
                color=colors.get(r['name'], 'gray'),
                density=True, histtype='stepfilled',
                label=f"{r['name']} benign (n={len(benign_scores)})")
ax.set_xlabel('production model score')
ax.set_ylabel('density')
ax.set_title('Benign score distributions across sets\n'
             '(rightward shift = model over-flags novel benign)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 0]
for r in results:
    if r['n'] == 0: continue
    valid = r['labels'] >= 0
    if valid.sum() < 30 or len(np.unique(r['labels'][valid])) < 2: continue
    y = r['labels'][valid]; sc = r['scores'][valid]
    fpr, tpr, _ = roc_curve(y, sc)
    auc = roc_auc_score(y, sc)
    ax.plot(fpr, tpr, color=colors.get(r['name'], 'gray'), lw=2,
            label=f"{r['name']} (AUC={auc:.3f}, n={int(valid.sum())})")
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.3)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC across sets, all lengths pooled')
ax.legend(loc='lower right', fontsize=10); ax.grid(alpha=0.3)

ax = axes[1, 1]
for r in results:
    if r['n'] == 0: continue
    color = colors.get(r['name'], 'gray')
    b_mask = r['labels'] == 0
    if b_mask.sum() == 0: continue
    if b_mask.sum() > 800:
        idx = np.random.RandomState(42).choice(np.where(b_mask)[0], 800, replace=False)
    else:
        idx = np.where(b_mask)[0]
    ax.scatter(r['gcs'][idx], r['scores'][idx], s=10, alpha=0.4,
               color=color, label=f"{r['name']} benign")
ax.axvspan(0, 0.35, alpha=0.08, color='orange', label='AT-bias region')
ax.set_xlabel('GC content of fragment')
ax.set_ylabel('production model score')
ax.set_title('GC vs score for benign fragments')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1)

fig.suptitle('Model evaluation: user / similar / external', fontsize=13, y=1.00)
fig.tight_layout()
main_plot = PLOT_DIR / 'evaluation.png'
fig.savefig(main_plot, dpi=110, bbox_inches='tight')
plt.close(fig)
print(f"  wrote {main_plot}")


print("\n" + "="*80)
print("EVALUATION SUMMARY")
print("="*80)
overall_aucs = {}
for r in results:
    if r['n'] == 0: continue
    valid = r['labels'] >= 0
    if valid.sum() > 0 and len(np.unique(r['labels'][valid])) >= 2:
        overall_aucs[r['name']] = roc_auc_score(r['labels'][valid], r['scores'][valid])
        print(f"  {r['name']:<14} overall AUC = {overall_aucs[r['name']]:.3f}  "
              f"(n={int(valid.sum())}, threat/benign={int(r['labels'][valid].sum())}/"
              f"{int((r['labels'][valid]==0).sum())})")

if 'similar' in overall_aucs and 'external' in overall_aucs:
    drift = overall_aucs['similar'] - overall_aucs['external']
    print(f"\n  similar -> external drop: {drift:+.3f}")
    if drift < 0.05:
        print(f"     model generalizes to phylogenetically distant organisms")
    elif drift < 0.15:
        print(f"     model has moderate distribution-specific bias")
    else:
        print(f"     model fails on novel organisms - retrain with broader data")

print("="*80)
print(f"\nAll outputs saved to: {Path(OUTPUT_DIR).resolve()}")
